# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
!git clone https://github.com/MarriamFatima-alt/flyrank-ml-internship.git
%cd flyrank-ml-internship
!python scripts/01_prepare_features.py
!python scripts/02_baseline_score.py
!python scripts/03_train_model.py
!python scripts/04_evaluate_and_export.py

import pandas as pd
queue = pd.read_csv("outputs/refresh_queue.csv")

print("Top 5 ranked actions:")
print(queue[["final_rank", "final_refresh_score", "suggested_action", "final_reason_codes"]].head(5).to_string(index=False))

print("\nAction mix:")
print(queue["suggested_action"].value_counts())

print("\nArchetype (position tier x trend) -> dominant action:")
archetype = queue.groupby(["position_tier", "trend_direction"])["suggested_action"].agg(lambda x: x.value_counts().idxmax())
print(archetype[archetype.index.get_level_values("trend_direction") == "down"])

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 133 (delta 48), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.86 MiB | 12.10 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/flyrank-ml-internship
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/outputs/model_results.json
Wrote final refresh queue: /content/flyrank-ml-internship/ou

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
print("Clients in training data:", queue["client_id"].nunique())
print("Rows scored:", len(queue))
print("Base rate (declining):", round(queue["is_declining_label"].mean(), 3))
print("Features used: no titles, URLs, or keywords -- confirmed by feature list in ml_utils.py")
print("Confidence tiers:", queue["confidence"].value_counts().to_dict())

Clients in training data: 32
Rows scored: 30000
Base rate (declining): 0.542
Features used: no titles, URLs, or keywords -- confirmed by feature list in ml_utils.py
Confidence tiers: {'low': 15000, 'medium': 11398, 'high': 3602}


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
low_conf = (queue["confidence"] == "low").sum()
print(f"Low-confidence rows requiring extra scrutiny: {low_conf:,} of {len(queue):,} ({low_conf/len(queue):.1%})")

expand_bucket = (queue["suggested_action"] == "expand_and_refresh").sum()
print(f"expand_and_refresh bucket size (too small to generalize from): {expand_bucket}")

Low-confidence rows requiring extra scrutiny: 15,000 of 30,000 (50.0%)
expand_and_refresh bucket size (too small to generalize from): 82


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
baseline_snapshot = {
    "precision_at_50_honest": 0.740,
    "base_rate": round(float(queue["is_declining_label"].mean()), 3),
    "action_mix": queue["suggested_action"].value_counts(normalize=True).round(3).to_dict(),
}
print("Baseline snapshot to monitor against next quarter:")
for k, v in baseline_snapshot.items():
    print(f"  {k}: {v}")

Baseline snapshot to monitor against next quarter:
  precision_at_50_honest: 0.74
  base_rate: 0.542
  action_mix: {'monitor': 0.436, 'refresh': 0.273, 'refresh_and_review_ctr': 0.222, 'refresh_and_review_engagement': 0.066, 'expand_and_refresh': 0.003}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import shutil, os

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/outputs/charts", exist_ok=True)

shutil.copy("outputs/refresh_queue.csv", "work/outputs/refresh_queue.csv")
shutil.copy("outputs/model_results.json", "work/outputs/model_results.json")
shutil.copy("outputs/summary.json", "work/outputs/summary.json")

for chart in ["action_mix.svg", "confidence_mix.svg", "top_reason_codes.svg", "top_feature_importance.svg", "trend_distribution.svg"]:
    shutil.copy(f"outputs/charts/{chart}", f"work/outputs/charts/{chart}")

print("Exported to work/outputs/:")
for f in os.listdir("work/outputs"):
    print(" ", f)

Exported to work/outputs/:
  charts
  summary.json
  model_results.json
  refresh_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.